## Chargement et préparation des données




In [ ]:


import scipy
import numpy as np  # N-dimensional arrays
import pandas as pd  # Data frames
import matplotlib.pyplot as plt  # Plotting
import seaborn  # Statistical visualizations
from sklearn import preprocessing  # Data preprocessing
# SciPy provides scientific computing routines.
import torch.nn as nn

import os

import torch
import random



In [ ]:
import scipy.sparse as sp
import scanpy as sc

adata = sc.read_h5ad("preprocessed.h5ad")
X = adata.X
if sp.issparse(X):
    X = X.toarray()
X = np.asarray(X, dtype = np.float32)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'preprocessed.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
X

In [ ]:
print(X.shape)
print(X.dtype)


In [ ]:
from torch.utils.data import DataLoader

X_tensor = torch.from_numpy(X).float()

loader = DataLoader(
    X_tensor,
    batch_size=128,
    shuffle=True
)

for batch in loader:
    print(batch.shape)
    break


## Structure du réseaux


In [ ]:

class Autoencodeur(nn.Module):
    def __init__(self,n,d):
        super().__init__()
        self.encodeur = nn.Sequential(
            nn.Linear(n,d),
            nn.Tanh(),
        )
        self.decodeur = nn.Sequential(
            nn.Linear(d,n),
        )



    def forward(self, x):
        x = self.encodeur(x)
        x = self.decodeur(x)
        return x

### Paramétrage


In [ ]:
import torch.optim as optim

n = X.shape[1]   # nombre de gènes = 18771
d = 64
net = Autoencodeur(n, d)

criterion = nn.MSELoss()
output_fn = nn.Sigmoid()
optimizer = optim.SGD(net.parameters(), lr=0.001)

### Valeur initial de la perte
#### Fait une premiere projection dans le reseau pour voir ce qui est reconstruit en sortie

In [ ]:
#projection avec les poids aléatoires
ZPred = net.forward(X_tensor)

#affichage
ZPred[:10,:]

In [ ]:
#valeur de la perte : observé vs. reconstitution
print(criterion(ZPred,X_tensor))


### on peu regarder les poid répartie aléatoirement dans la couche 1 (ici encodeur)

In [ ]:
net.encodeur[0].weight


### on peu regarder s'il y a une constante (biais) deja dans le réseaux


In [ ]:
net.encodeur[0].bias


# Entrainement du réseau

In [ ]:
import numpy
#fonction pour apprentissage avec les paramètres :
#Z_in, instance de classe torch, critère à optimiser, algo d'optimisation...
#...et n_epochs nombre de passage sur la base
def train_session(Z_in,modele,criterion,optimizer,n_epochs):
    #vecteur pour collecter le loss au fil des itérations
    losses = numpy.zeros(n_epochs)
    #itérer (boucle) pour optimiser - n_epochs fois sur la base
    for iter in range(n_epochs):
        #réinitialiser (ràz) le gradient
        #nécessaire à chaque passage sinon PyTorch accumule
        optimizer.zero_grad()
        #calculer la sortie du réseau
        Z_out = modele.forward(Z_in)
        #calculer la perte
        perte = criterion(Z_out,Z_in)
        #collecter la perte calculée dans le vecteur losses
        losses[iter] = perte.item()
        #calcul du gradient et retropropagation
        perte.backward()
        #màj des poids synaptiques
        optimizer.step()
    #sortie de la boucle
    #renvoie le vecteur avec les valeurs de perte à chaque epoch
    return losses

losses = train_session(X_tensor, net, criterion, optimizer, n_epochs=10000)


## visualisation de la perte

In [ ]:

plt.plot(losses)
plt.xlabel("époch")
plt.ylabel("MSE")
plt.show()

